# Reto SQL — Análisis de una temporada de Brisa United

## Contexto del problema

Trabajas como analista de datos de una liga de fútbol ficticia. El equipo **Brisa United** quiere un informe de su temporada a partir de los partidos registrados.

La pregunta principal es:

> **¿Cómo fueron los partidos de Brisa United y qué partidos explican mejor sus resultados, comparándolos con el contexto de la liga?**

El reto reúne en un mismo informe varias necesidades relacionadas: describir el resultado de cada partido, comparar sus goles con referencias de la liga, ordenar los partidos dentro de su contexto y seguir cómo evoluciona el rendimiento a medida que avanza la temporada.

La documentación de referencia trabaja con SQL, subconsultas, CTEs y funciones de ventana. El dialecto elegido para la solución es **PostgreSQL**, uno de los dialectos indicado en la documentación para las funciones de ventana.

## Datos disponibles

Todos los datos son sintéticos y están incluidos en este notebook.

La tabla `matches` contiene:

- `match_date`: fecha del partido.
- `region`: región de la competición.
- `home_team`: equipo local.
- `away_team`: equipo visitante.
- `home_goals`: goles del equipo local.
- `away_goals`: goles del equipo visitante.

Hay 16 partidos de una misma temporada. Brisa United aparece en 6 de ellos.

La información se muestra en formato tabular para que puedas comprobar los datos antes de empezar. La representación SQL equivalente también está incluida.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

teams = pd.DataFrame(
    [
        ("Aurora FC", "North"),
        ("Brisa United", "North"),
        ("Cumbres Athletic", "North"),
        ("Delta City", "South"),
        ("Estrella Real", "South"),
        ("Faro Sporting", "South"),
    ],
    columns=["team_name", "region"],
)

matches = pd.DataFrame(
    [
        ("2025-08-15", "North", "Aurora FC", "Brisa United", 2, 1),
        ("2025-08-16", "North", "Cumbres Athletic", "Delta City", 1, 1),
        ("2025-08-17", "North", "Brisa United", "Estrella Real", 0, 2),
        ("2025-08-18", "North", "Aurora FC", "Cumbres Athletic", 3, 0),
        ("2025-08-19", "South", "Delta City", "Faro Sporting", 2, 2),
        ("2025-08-20", "South", "Estrella Real", "Aurora FC", 1, 0),
        ("2025-08-21", "North", "Cumbres Athletic", "Brisa United", 2, 2),
        ("2025-08-22", "South", "Faro Sporting", "Aurora FC", 1, 3),
        ("2025-08-23", "North", "Brisa United", "Delta City", 1, 0),
        ("2025-08-24", "South", "Estrella Real", "Cumbres Athletic", 2, 1),
        ("2025-08-25", "North", "Aurora FC", "Delta City", 2, 2),
        ("2025-08-26", "South", "Faro Sporting", "Brisa United", 0, 1),
        ("2025-08-27", "North", "Cumbres Athletic", "Aurora FC", 1, 2),
        ("2025-08-28", "South", "Delta City", "Estrella Real", 3, 1),
        ("2025-08-29", "North", "Brisa United", "Aurora FC", 1, 1),
        ("2025-08-30", "South", "Faro Sporting", "Cumbres Athletic", 2, 0),
    ],
    columns=[
        "match_date", "region", "home_team", "away_team", "home_goals", "away_goals"
    ],
)
matches["match_date"] = pd.to_datetime(matches["match_date"])

display(teams)
display(matches)


,team_name,region
0,Aurora FC,North
1,Brisa United,North
2,Cumbres Athletic,North
3,Delta City,South
4,Estrella Real,South
5,Faro Sporting,South


,match_date,region,home_team,away_team,home_goals,away_goals
0,2025-08-15,North,Aurora FC,Brisa United,2,1
1,2025-08-16,North,Cumbres Athletic,Delta City,1,1
2,2025-08-17,North,Brisa United,Estrella Real,0,2
3,2025-08-18,North,Aurora FC,Cumbres Athletic,3,0
4,2025-08-19,South,Delta City,Faro Sporting,2,2
5,2025-08-20,South,Estrella Real,Aurora FC,1,0
6,2025-08-21,North,Cumbres Athletic,Brisa United,2,2
7,2025-08-22,South,Faro Sporting,Aurora FC,1,3
8,2025-08-23,North,Brisa United,Delta City,1,0
9,2025-08-24,South,Estrella Real,Cumbres Athletic,2,1


### Estructura SQL disponible

La solución debe trabajar con una tabla llamada `matches`. El siguiente DDL y los `INSERT` son la representación PostgreSQL de los mismos datos. No necesitas una fuente externa para disponer de ellos.

In [2]:
# Representación PostgreSQL de la preparación de los datos.
sql_setup = r"""CREATE TABLE matches (
    match_date DATE,
    region TEXT,
    home_team TEXT,
    away_team TEXT,
    home_goals INTEGER,
    away_goals INTEGER
);

INSERT INTO matches (
    match_date, region, home_team, away_team, home_goals, away_goals
)
VALUES
    ('2025-08-15', 'North', 'Aurora FC', 'Brisa United', 2, 1),
    ('2025-08-16', 'North', 'Cumbres Athletic', 'Delta City', 1, 1),
    ('2025-08-17', 'North', 'Brisa United', 'Estrella Real', 0, 2),
    ('2025-08-18', 'North', 'Aurora FC', 'Cumbres Athletic', 3, 0),
    ('2025-08-19', 'South', 'Delta City', 'Faro Sporting', 2, 2),
    ('2025-08-20', 'South', 'Estrella Real', 'Aurora FC', 1, 0),
    ('2025-08-21', 'North', 'Cumbres Athletic', 'Brisa United', 2, 2),
    ('2025-08-22', 'South', 'Faro Sporting', 'Aurora FC', 1, 3),
    ('2025-08-23', 'North', 'Brisa United', 'Delta City', 1, 0),
    ('2025-08-24', 'South', 'Estrella Real', 'Cumbres Athletic', 2, 1),
    ('2025-08-25', 'North', 'Aurora FC', 'Delta City', 2, 2),
    ('2025-08-26', 'South', 'Faro Sporting', 'Brisa United', 0, 1),
    ('2025-08-27', 'North', 'Cumbres Athletic', 'Aurora FC', 1, 2),
    ('2025-08-28', 'South', 'Delta City', 'Estrella Real', 3, 1),
    ('2025-08-29', 'North', 'Brisa United', 'Aurora FC', 1, 1),
    ('2025-08-30', 'South', 'Faro Sporting', 'Cumbres Athletic', 2, 0);"""
print(sql_setup)


CREATE TABLE matches (
    match_date DATE,
    region TEXT,
    home_team TEXT,
    away_team TEXT,
    home_goals INTEGER,
    away_goals INTEGER
);

INSERT INTO matches (
    match_date, region, home_team, away_team, home_goals, away_goals
)
VALUES
    ('2025-08-15', 'North', 'Aurora FC', 'Brisa United', 2, 1),
    ('2025-08-16', 'North', 'Cumbres Athletic', 'Delta City', 1, 1),
    ('2025-08-17', 'North', 'Brisa United', 'Estrella Real', 0, 2),
    ('2025-08-18', 'North', 'Aurora FC', 'Cumbres Athletic', 3, 0),
    ('2025-08-19', 'South', 'Delta City', 'Faro Sporting', 2, 2),
    ('2025-08-20', 'South', 'Estrella Real', 'Aurora FC', 1, 0),
    ('2025-08-21', 'North', 'Cumbres Athletic', 'Brisa United', 2, 2),
    ('2025-08-22', 'South', 'Faro Sporting', 'Aurora FC', 1, 3),
    ('2025-08-23', 'North', 'Brisa United', 'Delta City', 1, 0),
    ('2025-08-24', 'South', 'Estrella Real', 'Cumbres Athletic', 2, 1),
    ('2025-08-25', 'North', 'Aurora FC', 'Delta City', 2, 2),
    ('2025-08

## Objetivo

Construye **un único informe SQL** que permita analizar los 6 partidos de Brisa United.

El resultado final debe contener una fila por partido y permitir ver:

1. Fecha, región, condición de local/visitante y rival.
2. Goles marcados, goles recibidos y total de goles.
3. Resultado: victoria, derrota o empate.
4. Diferencia de goles.
5. Si el total de goles está por encima de la media de goles de su región.
6. Media total de goles de todos los partidos y media de su región.
7. Posición del partido dentro de su región según el total de goles.
8. Evolución acumulada de los goles marcados por Brisa.
9. Goles marcados por Brisa contando el partido actual y el inmediatamente anterior.
10. Una media móvil que considere el partido anterior, el actual y el siguiente cuando existan.
11. Goles marcados desde cada partido hasta el final de los partidos de Brisa, contando el partido actual.
12. Para las derrotas, una clasificación según la diferencia de goles en contra.

El informe debe conservar todos los partidos, incluidos los que no sean derrotas.

## Requisitos del reto

La solución debe demostrar, dentro del mismo informe:

- **CASE** para transformar y clasificar los datos.
- **Subconsultas** para añadir información sin perder el detalle de cada partido.
- Una subconsulta **correlacionada** que use la región del registro actual.
- Una subconsulta **anidada** que permita obtener una referencia regional.
- Al menos una **CTE** para organizar las etapas del resultado.
- Una función de ventana con **OVER**.
- **PARTITION BY** con una columna y también una partición por más de una dimensión relevante para el análisis.
- **RANK** para ordenar partidos y conservar los empates de rango.
- Ventanas deslizantes con `ROWS BETWEEN`.
- Un marco acumulado desde `UNBOUNDED PRECEDING` hasta `CURRENT ROW`.
- Un marco limitado con `PRECEDING`.
- Un marco que use `FOLLOWING`.
- Un marco que use `UNBOUNDED FOLLOWING`.

### Criterios de aceptación

- Aparece exactamente una fila por partido de Brisa United.
- Las fechas quedan ordenadas de la más antigua a la más reciente.
- Las derrotas se identifican sin ambigüedad.
- La primera posición de la clasificación de derrotas corresponde a la mayor diferencia de goles en contra.
- Las comparaciones regionales usan los datos de toda la competición, no solo los partidos de Brisa.
- Los cálculos que dependen del orden temporal respetan la fecha.
- No hacen falta descargas, APIs, Internet ni bases de datos externas.

## Espacio para la resolución del usuario

Escribe y prueba aquí tu propia solución. Puedes modificar las celdas tantas veces como necesites.

Antes de continuar, intenta conseguir todos los criterios de aceptación sin consultar la solución.

In [3]:
# Tu solución SQL para el reto.
user_solution = ""


In [4]:
# Espacio adicional de pruebas.


# Solución - consultar después de intentar

La siguiente consulta está escrita en sintaxis PostgreSQL y resuelve el reto completo.

In [5]:
solution_sql = r'''WITH match_scope AS (
    SELECT
        m.match_date,
        m.region,
        m.home_team,
        m.away_team,
        m.home_goals,
        m.away_goals,
        m.home_goals + m.away_goals AS total_goals
    FROM matches AS m
    WHERE m.home_team = 'Brisa United'
       OR m.away_team = 'Brisa United'
),
brisa_base AS (
    SELECT
        ms.*,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN 'HOME'
            ELSE 'AWAY'
        END AS venue,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN ms.away_team
            ELSE ms.home_team
        END AS opponent,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN ms.home_goals
            ELSE ms.away_goals
        END AS goals_for,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN ms.away_goals
            ELSE ms.home_goals
        END AS goals_against
    FROM match_scope AS ms
),
brisa_metrics AS (
    SELECT
        bb.*,
        bb.goals_for - bb.goals_against AS goal_margin,
        CASE
            WHEN bb.goals_for > bb.goals_against THEN 'WIN'
            WHEN bb.goals_for < bb.goals_against THEN 'LOSS'
            ELSE 'DRAW'
        END AS outcome,
        (SELECT AVG(m.total_goals) FROM matches AS m) AS overall_avg_goals,
        (
            SELECT AVG(m2.total_goals)
            FROM matches AS m2
            WHERE m2.region = bb.region
        ) AS region_avg_goals,
        CASE
            WHEN bb.total_goals > (
                SELECT AVG(region_rows.total_goals)
                FROM (
                    SELECT m3.total_goals
                    FROM matches AS m3
                    WHERE m3.region = bb.region
                ) AS region_rows
            )
            THEN 'ABOVE_REGION_AVG'
            ELSE 'AT_OR_BELOW_REGION_AVG'
        END AS region_goal_flag
    FROM brisa_base AS bb
),
ranked AS (
    SELECT
        bm.*,
        RANK() OVER (
            PARTITION BY bm.region
            ORDER BY bm.total_goals DESC
        ) AS region_goal_rank,
        AVG(bm.goals_for) OVER (
            PARTITION BY bm.region, bm.venue
        ) AS region_venue_avg_goals_for,
        SUM(bm.goals_for) OVER (
            ORDER BY bm.match_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_goals_for,
        SUM(bm.goals_for) OVER (
            ORDER BY bm.match_date
            ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
        ) AS last_two_goals_for,
        AVG(bm.total_goals) OVER (
            ORDER BY bm.match_date
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS moving_avg_three_matches,
        SUM(bm.goals_for) OVER (
            ORDER BY bm.match_date
            ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
        ) AS future_goals_for_inclusive
    FROM brisa_metrics AS bm
),
loss_ranked AS (
    SELECT
        r.*,
        RANK() OVER (ORDER BY (r.goals_against - r.goals_for) DESC) AS loss_rank
    FROM ranked AS r
    WHERE r.outcome = 'LOSS'
)
SELECT
    r.match_date,
    r.region,
    r.venue,
    r.opponent,
    r.goals_for,
    r.goals_against,
    r.total_goals,
    r.outcome,
    r.goal_margin,
    r.region_goal_flag,
    r.overall_avg_goals,
    r.region_avg_goals,
    r.region_goal_rank,
    r.region_venue_avg_goals_for,
    r.running_goals_for,
    r.last_two_goals_for,
    r.moving_avg_three_matches,
    r.future_goals_for_inclusive,
    lr.loss_rank
FROM ranked AS r
LEFT JOIN loss_ranked AS lr
    ON lr.match_date = r.match_date
ORDER BY r.match_date;'''
print(solution_sql)


WITH match_scope AS (
    SELECT
        m.match_date,
        m.region,
        m.home_team,
        m.away_team,
        m.home_goals,
        m.away_goals,
        m.home_goals + m.away_goals AS total_goals
    FROM matches AS m
    WHERE m.home_team = 'Brisa United'
       OR m.away_team = 'Brisa United'
),
brisa_base AS (
    SELECT
        ms.*,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN 'HOME'
            ELSE 'AWAY'
        END AS venue,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN ms.away_team
            ELSE ms.home_team
        END AS opponent,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN ms.home_goals
            ELSE ms.away_goals
        END AS goals_for,
        CASE
            WHEN ms.home_team = 'Brisa United' THEN ms.away_goals
            ELSE ms.home_goals
        END AS goals_against
    FROM match_scope AS ms
),
brisa_metrics AS (
    SELECT
        bb.*,
        bb.goals_for - bb.goals_against AS goa

## Resultados obtenidos

El entorno usado para generar y validar este notebook no dispone de un servidor PostgreSQL local. Para mantener el notebook autocontenido y no instalar paquetes durante su ejecución, la comprobación de los resultados se realiza con un **ejecutor de referencia en Python** sobre los mismos datos sintéticos.

Esta comprobación valida desde cero los valores que debe producir la consulta PostgreSQL. La consulta SQL anterior sigue siendo la solución; la implementación Python se utiliza únicamente para verificar los resultados del notebook.

In [6]:
# Ejecutor de referencia de los resultados esperados.
target = "Brisa United"

base = matches[
    (matches["home_team"] == target) | (matches["away_team"] == target)
].copy()

base["venue"] = np.where(base["home_team"].eq(target), "HOME", "AWAY")
base["opponent"] = np.where(
    base["home_team"].eq(target), base["away_team"], base["home_team"]
)
base["goals_for"] = np.where(
    base["home_team"].eq(target), base["home_goals"], base["away_goals"]
)
base["goals_against"] = np.where(
    base["home_team"].eq(target), base["away_goals"], base["home_goals"]
)
base["total_goals"] = base["goals_for"] + base["goals_against"]
base["goal_margin"] = base["goals_for"] - base["goals_against"]
base["outcome"] = np.select(
    [base["goal_margin"] > 0, base["goal_margin"] < 0],
    ["WIN", "LOSS"],
    default="DRAW",
)
base = base.sort_values("match_date").reset_index(drop=True)

all_total = matches["home_goals"] + matches["away_goals"]
overall_avg = all_total.mean()
region_avg = matches.assign(total_goals=all_total).groupby("region")["total_goals"].mean()

base["overall_avg_goals"] = overall_avg
base["region_avg_goals"] = base["region"].map(region_avg)
base["region_goal_flag"] = np.where(
    base["total_goals"] > base["region_avg_goals"],
    "ABOVE_REGION_AVG",
    "AT_OR_BELOW_REGION_AVG",
)
base["region_goal_rank"] = (
    base.groupby("region")["total_goals"].rank(method="min", ascending=False).astype(int)
)

team_perspective = []
for row in matches.itertuples(index=False):
    team_perspective.append((row.match_date, row.region, "HOME", row.home_team, int(row.home_goals)))
    team_perspective.append((row.match_date, row.region, "AWAY", row.away_team, int(row.away_goals)))

team_perspective = pd.DataFrame(
    team_perspective,
    columns=["match_date", "region", "venue", "team", "goals_for"]
)
region_venue_avg = team_perspective.groupby(["region", "venue"])["goals_for"].mean()

base["region_venue_avg_goals_for"] = [
    region_venue_avg[(region, venue)]
    for region, venue in zip(base["region"], base["venue"])
]

base["running_goals_for"] = base["goals_for"].cumsum()
base["last_two_goals_for"] = base["goals_for"].rolling(2, min_periods=1).sum()

values = base["total_goals"].tolist()
base["moving_avg_three_matches"] = [
    sum(values[max(0, i - 1):min(len(values), i + 2)])
    / len(values[max(0, i - 1):min(len(values), i + 2)])
    for i in range(len(values))
]
base["future_goals_for_inclusive"] = (
    base["goals_for"].iloc[::-1].cumsum().iloc[::-1].to_numpy()
)

losses = base[base["outcome"].eq("LOSS")].copy()
losses["loss_margin"] = losses["goals_against"] - losses["goals_for"]
losses["loss_rank"] = losses["loss_margin"].rank(method="min", ascending=False).astype(int)

base["loss_rank"] = base["match_date"].map(
    losses.set_index("match_date")["loss_rank"]
)

result = base[
    [
        "match_date", "region", "venue", "opponent", "goals_for",
        "goals_against", "total_goals", "outcome", "goal_margin",
        "region_goal_flag", "overall_avg_goals", "region_avg_goals",
        "region_goal_rank", "region_venue_avg_goals_for",
        "running_goals_for", "last_two_goals_for",
        "moving_avg_three_matches", "future_goals_for_inclusive", "loss_rank"
    ]
].copy()

display(result.round(4))


,match_date,region,venue,opponent,goals_for,goals_against,total_goals,outcome,goal_margin,region_goal_flag,overall_avg_goals,region_avg_goals,region_goal_rank,region_venue_avg_goals_for,running_goals_for,last_two_goals_for,moving_avg_three_matches,future_goals_for_inclusive,loss_rank
0,2025-08-15,North,AWAY,Aurora FC,1,2,3,LOSS,-1,ABOVE_REGION_AVG,2.6875,2.6667,2,1.2222,1,1.0,2.5000,6,2.0
1,2025-08-17,North,HOME,Estrella Real,0,2,2,LOSS,-2,AT_OR_BELOW_REGION_AVG,2.6875,2.6667,3,1.4444,1,1.0,3.0000,5,1.0
2,2025-08-21,North,AWAY,Cumbres Athletic,2,2,4,DRAW,0,ABOVE_REGION_AVG,2.6875,2.6667,1,1.2222,3,2.0,2.3333,5,NaN
3,2025-08-23,North,HOME,Delta City,1,0,1,WIN,1,AT_OR_BELOW_REGION_AVG,2.6875,2.6667,5,1.4444,4,3.0,2.0000,3,NaN
4,2025-08-26,South,AWAY,Faro Sporting,1,0,1,WIN,1,AT_OR_BELOW_REGION_AVG,2.6875,2.7143,1,1.1429,5,2.0,1.3333,2,NaN
5,2025-08-29,North,HOME,Aurora FC,1,1,2,DRAW,0,AT_OR_BELOW_REGION_AVG,2.6875,2.6667,3,1.4444,6,2.0,1.5000,1,NaN


In [7]:
# Validación automática de los resultados obtenidos.
assert len(result) == 6
assert result["match_date"].is_monotonic_increasing
assert result["match_date"].is_unique
assert result["outcome"].tolist() == ["LOSS", "LOSS", "DRAW", "WIN", "WIN", "DRAW"]

loss_check = result[result["outcome"] == "LOSS"][["opponent", "loss_rank"]]
assert dict(zip(loss_check["opponent"], loss_check["loss_rank"])) == {
    "Aurora FC": 2,
    "Estrella Real": 1,
}

assert round(float(result["overall_avg_goals"].iloc[0]), 4) == 2.6875
assert round(float(result.loc[result["region"] == "North", "region_avg_goals"].iloc[0]), 4) == 2.6667
assert round(float(result.loc[result["region"] == "South", "region_avg_goals"].iloc[0]), 4) == 2.7143

assert result["running_goals_for"].tolist() == [1, 1, 3, 4, 5, 6]
assert result["last_two_goals_for"].tolist() == [1.0, 1.0, 2.0, 3.0, 2.0, 2.0]
assert result["future_goals_for_inclusive"].tolist() == [6, 5, 5, 3, 2, 1]
assert round(float(result["region_venue_avg_goals_for"].iloc[0]), 4) == 1.2222
assert round(float(result["region_venue_avg_goals_for"].iloc[1]), 4) == 1.4444
assert round(float(result["region_venue_avg_goals_for"].iloc[4]), 4) == 1.1429

print("Todas las comprobaciones de resultados han sido superadas.")


Todas las comprobaciones de resultados han sido superadas.


## Interpretación y autoevaluación

La solución separa el problema en etapas. Primero delimita los partidos de Brisa y deriva la información necesaria para describirlos. Después añade referencias globales y regionales mediante subconsultas y organiza el resultado mediante CTEs. Finalmente, las funciones de ventana permiten calcular posiciones, acumulados y marcos móviles sin perder la fila original de cada partido.

Para autoevaluarte, compara tu consulta con la solución y revisa:

- si llegaste al mismo conjunto de 6 partidos;
- si tus `CASE` clasifican correctamente los resultados;
- si tus subconsultas mantienen el detalle de cada partido;
- si tus CTEs hacen más clara la construcción del resultado;
- si tus particiones usan el contexto que necesita cada cálculo;
- si el orden temporal de las ventanas es correcto;
- si distingues una ventana acumulada de una ventana limitada;
- si la clasificación de las derrotas conserva los empates de `RANK`.

Una solución diferente puede ser válida si produce los mismos resultados y demuestra de forma relevante los contenidos de la documentación.